# Notebook 01 — Carga, parseo y depuracion

Prototipo funcional de la Evaluacion 1. Este notebook deja el dataset en un estado **reproducible, justificado y cientificamente defendible**, siguiendo el flujo:

```
Leer -> Parsear -> Depurar -> Imputar/eliminar
```

**Dataset:** openfootball/football.json — Premier League 2024/25.

## 0. Configuracion

Ruta al repo football.json. Ajustar `DATA_DIR` si se corre en otra maquina.

In [ ]:
import sys
from pathlib import Path

# Raiz del proyecto (para importar src/)
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / 'src'))

# Ruta al dataset football.json (no se duplica en el repo)
DATA_DIR = Path(r'C:/Users/1234/Documents/ITM/4 semestre/programacion_cientifica/football.json')
ARCHIVO = DATA_DIR / '2024-25' / 'en.1.json'

print('Dataset:', ARCHIVO)
print('Existe:', ARCHIVO.exists())

## 1. Leer (I/O)

Se carga el JSON crudo a un DataFrame plano. Los goles quedan como numero; los `null` del JSON se conservan como `NaN` reales (no se inventan valores en esta etapa).

In [ ]:
import pandas as pd
from data_loader import cargar_partidos

df_raw = cargar_partidos(ARCHIVO, temporada='2024-25')
print('Shape:', df_raw.shape)
print('Columnas:', df_raw.columns.tolist())
df_raw.head()

In [ ]:
# Inspeccion rapida de tipos crudos (antes de parsear)
df_raw.dtypes

## 2. Parsear (tipo)

Se interpretan los valores crudos como tipos con semantica: fecha a datetime, goles a entero nullable, texto normalizado.

In [ ]:
from data_clean import parsear_tipos

df = parsear_tipos(df_raw)
df.dtypes

In [ ]:
# Contar cuantos NaN nacieron en el parseo vs los que ya venian del JSON.
# En este dataset los NaN de goles vienen del JSON (null), no del parseo.
print('Faltantes despues de parsear:')
print(df[['ht_g1','ht_g2','ft_g1','ft_g2']].isna().sum())

## 3. Depurar (calidad)

### 3.1 Auditoria minima

Reporte de dtype, faltantes, unicos y duplicados.

In [ ]:
from data_clean import auditoria

reporte = auditoria(df)
reporte

### 3.2 Reglas de dominio

Se marcan las filas que rompen reglas de dominio (no se borran a ciegas, se senalan):

- goles negativos (imposible)
- ft < ht (el tiempo completo no puede tener menos goles que el medio tiempo)
- team1 == team2 (un equipo no juega contra si mismo)
- fecha invalida
- sin ft (partido sin resultado final)
- sin ht (medio tiempo no registrado)

In [ ]:
from data_clean import marcar_inconsistencias

df_marcado = marcar_inconsistencias(df)
cols_flag = ['imp_goles_neg','imp_ft_menor_ht','imp_mismo_equipo','imp_fecha_invalida','sin_ft','sin_ht']
print('Conteo de inconsistencias:')
print(df_marcado[cols_flag].sum())

### 3.3 Decisiones de depuracion

1. **Eliminar filas sin ft**: un partido sin resultado final no aporta al analisis ht vs ft. Informacion no recuperable.
2. **Conservar filas sin ht pero con ft**: el medio tiempo falta, pero el resultado final sirve. Se excluyen del scatter ht vs ft pero no se borran.
3. **Eliminar inconsistencias imposibles** (goles negativos, ft < ht, mismo equipo): errores de captura.
4. **Eliminar duplicados exactos**.

In [ ]:
from data_clean import depurar

df_limpio = depurar(df, eliminar_sin_ft=True)
print('Shape final:', df_limpio.shape)
print('Shape inicial:', df.shape)
print('Filas eliminadas:', df.shape[0] - df_limpio.shape[0])

## 4. Derivar variables para el analisis

Columnas utiles para el punto 5 (dinamica ht vs ft): goles totales, resultado en cada momento, indicador de remontada.

In [ ]:
from data_clean import derivar_resultado

df_limpio = derivar_resultado(df_limpio)
df_limpio[['team1','team2','ht_g1','ht_g2','ft_g1','ft_g2','ht_total','ft_total','res_ht','res_ft','remontada']].head(10)

In [ ]:
# Resumen rapido: cuantas remontadas hay en la temporada
print('Partidos con ht registrado:', df_limpio['res_ht'].notna().sum())
print('Remontadas:', df_limpio['remontada'].sum())
print('% remontadas:', round(df_limpio['remontada'].mean()*100, 2), '%')

## 5. Guardar dataset procesado

Se exporta el dataset limpio a `data/processed/` para que el notebook 02 lo consuma.

In [ ]:
out = PROJECT_DIR / 'data' / 'processed' / 'premier_2024_25_limpio.csv'
df_limpio.to_csv(out, index=False)
print('Guardado en:', out)
print('Filas:', len(df_limpio))
print('Columnas:', df_limpio.shape[1])

## 6. Conclusion del prototipo

- El dataset se carga de forma reproducible desde el JSON.
- Se parsean tipos (fecha, goles nullable, texto normalizado).
- Se depura con reglas de dominio justificadas, no con borrado automatico de atipicos.
- Quedan 380 partidos (Premier 2024-25) con resultado final; ~16 sin ht registrado.
- Primer resultado funcional: tabla limpia + indicador de remontadas.

**Siguiente:** `02_analisis_dinamica_ht_vs_ft.ipynb` para las visualizaciones del punto 5.